# AI-Accelerated QEC for Neutral-Atom Logical Qubits

This interactive companion to Infleqtion’s [AI-accelerated QEC post](https://infleqtion.com/ai-accelerated-qec/) turns its leakage-aware NVIDIA Ising Decoding experiment into runnable cells. It keeps the experiment deliberately transparent: change the leakage probabilities, number of shots, and execution mode, regenerate the metrics, and inspect the resulting logical-error-rate and decoder-latency plots.

> **Heuristic simulation.** This notebook injects leakage transitions after two-qubit Clifford gates on top of a fixed circuit-level Pauli noise model for a distance-9 surface-code memory experiment. It is an exploration tool, not a hardware-calibrated prediction.

## Why decoding matters

QEC is both a quantum and a classical-computing problem. Every correction round creates syndrome data that must be processed fast enough to keep pace with the hardware. NVIDIA Ising Decoding provides a learned GPU-resident pass that sparsifies syndrome information before downstream decoding, reducing the classical burden while retaining information needed for correction.

For neutral atoms, this is especially relevant as measurement and control loops become faster: useful logical qubits require accuracy, throughput, and low decoding latency together.

In [8]:
import sys
import torch

## Neutral atoms are qubits in theory, but qudits in practice

Neutral-atom hardware can occupy levels outside the computational subspace. In this simplified model, `|0L⟩` and `|1L⟩` leak out of the computational space but are still read as 0-like and 1-like outcomes. The decoder therefore sees leakage indirectly through a syndrome stream shaped by both Pauli-type noise and out-of-subspace population.

Run the next cell to explore that binary readout mapping.

In [9]:
def map_readout_state(level: int) -> int:
    """Map computational and leakage levels to a binary readout."""
    zero_like = {0, 2}  # |0⟩ and |0L⟩
    one_like = {1, 3}   # |1⟩ and |1L⟩
    if level in zero_like:
        return 0
    if level in one_like:
        return 1
    raise ValueError(f"Unexpected state label: {level}")

for level, label in enumerate(("|0⟩", "|1⟩", "|0L⟩", "|1L⟩")):
    print(f"{label:4} → observed binary outcome {map_readout_state(level)}")

|0⟩  → observed binary outcome 0
|1⟩  → observed binary outcome 1
|0L⟩ → observed binary outcome 0
|1L⟩ → observed binary outcome 1


## Experiment setup

This tutorial is run from the checked-out repository, including `third_party/ising-decoding`. The local model is NVIDIA's public pretrained Chamberland `PreDecoderModelMemory_v1` checkpoint, stored through Git LFS. Install the tutorial dependencies in the active environment:

```bash
%pip install numpy torch pymatching deltakit-stim
```

Install Git LFS before cloning the repository so the checkpoint is a real PyTorch file rather than an LFS pointer. This notebook uses **Deltakit-Stim**, not `leaky`: it converts detected leakage into heralded-erasure flags. The frozen Chamberland CNN consumes only its original four syndrome channels; the residual syndrome, erasure flags, and raw readout features are passed to the global neural decoder. A GPU is recommended for the full dataset and MLP training.

In [10]:
import json
import os
import re
import subprocess
import sys
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

mpl.rcParams.update({"font.size": 12, "axes.titlesize": 13, "axes.labelsize": 12})
%config InlineBackend.figure_format = 'retina'

In [11]:
repo_dir = Path.cwd().resolve()
ising_dir = repo_dir / "third_party/ising-decoding"
scripts_dir = ising_dir / "code/scripts"
checkpoint = ising_dir / "models/Ising-Decoder-SurfaceCode-1-Fast.pt"
data_dir = repo_dir / "data"
output_dir = repo_dir / "outputs"
if not checkpoint.exists():
    raise RuntimeError("Missing Chamberland checkpoint. Run `git lfs install` then `git lfs pull` in third_party/ising-decoding.")
print(f"Using frozen Chamberland checkpoint: {checkpoint}")
print(f"Deltakit/NN tutorial scripts: {scripts_dir}")

Using frozen Chamberland checkpoint: /home/ubuntu/Ieee_qec_26_tutorial/third_party/ising-decoding/models/Ising-Decoder-SurfaceCode-1-Fast.pt
Deltakit/NN tutorial scripts: /home/ubuntu/Ieee_qec_26_tutorial/third_party/ising-decoding/code/scripts


## Run controls

The notebook trains live by default on CUDA. Each run creates fresh Deltakit-Stim training and held-out shards, freezes the pretrained Chamberland local CNN, trains a new global MLP, evaluates it, and plots the live held-out result. Select smoke only to test the wiring quickly.

In [12]:
# Run these cells in order on a CUDA kernel.
RUN_PIPELINE = True
EXECUTION_MODE = "gpu"  # choose "smoke" or "gpu"
RUN_SEED = None  # None creates a fresh random training/held-out split; set an integer to reproduce it.

if EXECUTION_MODE == "smoke":
    SHOTS, EPOCHS, BATCH_SIZE = 4_096, 3, 256
elif EXECUTION_MODE == "gpu":
    SHOTS, EPOCHS, BATCH_SIZE = 65_536, 20, 256
else:
    raise ValueError("EXECUTION_MODE must be 'smoke' or 'gpu'.")

print({"mode": EXECUTION_MODE, "shots": SHOTS, "epochs": EPOCHS, "batch_size": BATCH_SIZE, "seed": RUN_SEED})

{'mode': 'gpu', 'shots': 65536, 'epochs': 20, 'batch_size': 256, 'seed': None}


In [14]:
# This is the end-to-end AI path used by the tutorial.
# Deltakit-Stim supplies the herald map; the frozen Chamberland CNN never sees it.
# The global MLP receives Chamberland's residual syndrome, herald map, and readout features.
if RUN_PIPELINE:
    runner_python = Path(sys.executable).resolve()
    run_seed = int(np.random.SeedSequence().generate_state(1)[0]) if RUN_SEED is None else int(RUN_SEED)
    print(f"Generating a fresh Deltakit-Stim split with seed={run_seed}.")
    dependency_check = subprocess.run(
        [str(runner_python), "-c", "import torch, pymatching, deltakit_stim; assert torch.cuda.is_available(), 'CUDA is required for this live tutorial run'"],
        text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    if dependency_check.returncode:
        raise RuntimeError("Install torch, pymatching, and deltakit-stim, then restart the kernel.\n" + dependency_check.stdout)

    suffix = f"d9_{EXECUTION_MODE}"
    circuit_shard = data_dir / f"loss_aware_{suffix}_train.npz"
    heldout_circuit_shard = data_dir / f"loss_aware_{suffix}_heldout.npz"
    residual_shard = data_dir / f"bluvstein_residual_{suffix}_train.npz"
    heldout_residual_shard = data_dir / f"bluvstein_residual_{suffix}_heldout.npz"
    global_checkpoint = output_dir / f"bluvstein_global_{suffix}.pt"
    commands = [
        [str(runner_python), str(scripts_dir / "generate_loss_aware_dataset.py"),
         "--distance", "9", "--rounds", "9", "--shots", str(SHOTS),
         "--pauli-p", "0.002", "--erasure-p", "0.01", "--seed", str(run_seed), "--teacher", "circuit-frame",
         "--output", str(circuit_shard)],
        [str(runner_python), str(scripts_dir / "generate_loss_aware_dataset.py"),
         "--distance", "9", "--rounds", "9", "--shots", str(max(512, SHOTS // 8)),
         "--pauli-p", "0.002", "--erasure-p", "0.01", "--seed", str(run_seed + 1), "--teacher", "circuit-frame",
         "--output", str(heldout_circuit_shard)],
        [str(runner_python), str(scripts_dir / "generate_bluzstein_residual_dataset.py"),
         str(circuit_shard), str(checkpoint), "--device", "cuda", "--batch-size", str(BATCH_SIZE),
         "--balance-logical-labels", "--seed", str(run_seed), "--output", str(residual_shard)],
        [str(runner_python), str(scripts_dir / "generate_bluzstein_residual_dataset.py"),
         str(heldout_circuit_shard), str(checkpoint), "--device", "cuda", "--batch-size", str(BATCH_SIZE),
         "--balance-logical-labels", "--seed", str(run_seed + 1), "--output", str(heldout_residual_shard)],
        [str(runner_python), str(scripts_dir / "train_bluvstein_global_decoder.py"),
         str(residual_shard), "--device", "cuda", "--batch-size", str(BATCH_SIZE), "--epochs", str(EPOCHS),
         "--output", str(global_checkpoint)],
        [str(runner_python), str(scripts_dir / "evaluate_bluvstein_global_decoder.py"),
         str(heldout_residual_shard), str(global_checkpoint), "--device", "cuda"],
    ]
    evaluation_stdout = None
    for command in commands:
        print("$", " ".join(command))
        completed = subprocess.run(command, cwd=repo_dir, check=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        print(completed.stdout, end="")
        if command[1].endswith("evaluate_bluvstein_global_decoder.py"):
            evaluation_stdout = completed.stdout

    metrics = dict(re.findall(r"^(BCE|global residual accuracy|final logical LER): ([0-9.]+)$", evaluation_stdout, flags=re.MULTILINE))
    if len(metrics) != 3:
        raise RuntimeError("Could not parse live evaluator output:\n" + evaluation_stdout)
    result = {
        "dataset": f"fresh Deltakit-Stim d=9 shard ({SHOTS:,} training shots; seed={run_seed})",
        "global_decoder": "newly trained Bluvstein-style MLP after frozen Chamberland CNN",
        "heldout_bce": float(metrics["BCE"]),
        "heldout_global_accuracy": float(metrics["global residual accuracy"]),
        "heldout_logical_failure": float(metrics["final logical LER"]),
    }
else:
    print("Set RUN_PIPELINE = True to generate a Deltakit-Stim shard and train the global MLP.")

Generating a fresh Deltakit-Stim split with seed=2477307500.


RuntimeError: Install torch, pymatching, and deltakit-stim, then restart the kernel.
Traceback (most recent call last):
  File "<string>", line 1, in <module>
ModuleNotFoundError: No module named 'torch'


## Results: loss-aware AI decoding

The implemented path is Deltakit-Stim → frozen Chamberland local CNN → Bluvstein-style global MLP. Deltakit-Stim converts the leakage event into a known erasure location. The Chamberland model processes only its original four syndrome channels and returns a residual syndrome plus a local logical frame. The global MLP receives that residual, the herald map, and raw measurement/readout features, then predicts the remaining logical parity.

The final logical sign is S_L = S_L^(local) XOR S_L^(global). This MLP is the tutorial's global AI decoder; loss-aware PyMatching remains a useful exact, deterministic reference baseline, but is not the downstream stage in this pipeline. The plot below uses the newly trained model's balanced held-out logical-classification result, not a physical-error-rate threshold claim.

In [ ]:
# `result` is created by the live evaluation cell above; no stored data are loaded here.
# The software logical flip balances the binary global-label task, so chance accuracy is 50%.
if "result" not in globals():
    raise RuntimeError("Run the live pipeline cell first so this plot uses a newly trained decoder.")
for name, value in result.items():
    print(f"{name}: {value}")

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")
labels = ["Held-out global\naccuracy", "Held-out logical\nfailure"]
values = [result["heldout_global_accuracy"], result["heldout_logical_failure"]]
colors = ["tab:green", "tab:red"]
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(labels, values, color=colors)
ax.bar_label(bars, labels=[f"{value:.3%}" for value in values], padding=3)
ax.set_ylim(0, 1)
ax.set_ylabel("Fraction of balanced held-out shots")
ax.set_title("Deltakit-Stim + Chamberland local CNN + global MLP")
ax.text(0.5, -0.22, "Prototype result: it validates the loss-aware AI interface; it is not a threshold or PyMatching comparison.",
        ha="center", va="top", transform=ax.transAxes, fontsize=10)
fig.tight_layout()
plt.show()

## From leakage to erasure

Leakage is not always an opaque error channel. Here, Deltakit-Stim turns each detected leakage event into a heralded-erasure flag. The flag does not reveal which Pauli occurred, but it tells the decoder where the uncertainty is concentrated. It is therefore passed unchanged to the global neural decoder alongside the Chamberland-predecoded residual.

This is deliberately not a retrained loss-aware Chamberland local CNN: a detected loss/reset has no unique local Pauli-correction label. Instead, the pretrained local CNN keeps its established task, while the global MLP learns the unambiguous remaining logical parity using both residual syndrome and erasure context. A herald-conditioned PyMatching decoder is retained in the repository as an optional correctness/reference baseline.

## Why this matters for Sqale

The broader objective is a coherent hybrid QPU–GPU stack: strong physical qubits, scalable logical architectures, GPU-resident classical acceleration, and AI models that make control and decoding faster and more informed. For neutral atoms, Deltakit-Stim supplies a simple circuit-level heralded-leakage model, while the two-stage neural decoder illustrates local preprocessing followed by global loss-aware inference.

Read the full [published post](https://infleqtion.com/ai-accelerated-qec/) for the neutral-atom context, platform roadmap, and source figures. This notebook is the executable companion for the loss-aware AI-decoding experiment.